# FiftyOne Quickstart

Hello there! This notebook provides a brief walkthrough of [FiftyOne](https://voxel51.com/docs/fiftyone), highlighting features that will help you build better datasets and computer vision models.

We'll cover the following concepts:

- Loading a dataset [into FiftyOne](https://voxel51.com/docs/fiftyone/user_guide/dataset_creation/index.html)
- Using FiftyOne [in a notebook](https://voxel51.com/docs/fiftyone/environments/index.html#notebooks)
- Using [views](https://voxel51.com/docs/fiftyone/user_guide/using_views.html) and [the App](https://voxel51.com/docs/fiftyone/user_guide/app.html) to explore different aspects of your dataset
- [Evaluating](https://voxel51.com/docs/fiftyone/user_guide/evaluation.html) your model's predictions
- [Finding label mistakes](https://voxel51.com/docs/fiftyone/user_guide/brain.html#label-mistakes) in your datasets

## Install FiftyOne


In [ ]:
!pip install fiftyone

If you're running this in Google Colab (or any other Ubuntu 22.04 machine), you'll also need to install:

In [ ]:
!pip install fiftyone-db-ubuntu2204

## Load a dataset

Let's get started by importing the FiftyOne library:

In [ ]:
import fiftyone as fo

FiftyOne provides a number of helpful data/model resources to get you up and running on your projects. In this example, we'll load a small detection dataset from the [FiftyOne Dataset Zoo](https://voxel51.com/docs/fiftyone/user_guide/dataset_zoo/index.html).

The command below downloads the dataset from the web and loads it into a [FiftyOne Dataset](https://voxel51.com/docs/fiftyone/user_guide/basics.html) that we'll use to explore the capabilities of FiftyOne:

In [ ]:
import fiftyone.zoo as foz

dataset = foz.load_zoo_dataset("coco-2014", split='validation')

In [ ]:
print(dataset)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# !cp /root/fiftyone/coco-2014/validation/labels.json ./
# !cp -r /content/drive/MyDrive/coco2014_validation_labels.json ./
# !cp -r /content/drive/MyDrive/karpathy_test_images/ ./

Now let's launch the [FiftyOne App](https://voxel51.com/docs/fiftyone/user_guide/app.html) so we can explore the dataset visually. Right away you will see that because we are in a notebook, an embedded instance of the App with our dataset loaded has been rendered in the cell's output.

The [Session](https://voxel51.com/docs/fiftyone/api/fiftyone.core.session.html#fiftyone.core.session.Session) object created below is a bi-directional connection between your Python kernel and the FiftyOne App, as we'll see later.

In [ ]:
session = fo.launch_app(dataset)

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz
from fiftyone import ViewField as F
import json
import numpy as np
import os

bbox_area = F("bounding_box")[2] * F("bounding_box")[3]
bbox_area0 = F("bounding_box")[0]
bbox_area1 = F("bounding_box")[1]
bbox_area2 = F("bounding_box")[2]
bbox_area3 = F("bounding_box")[3]
!wget https://storage.googleapis.com/sfr-vision-language-research/datasets/coco_karpathy_test.json
ann = json.load(open('coco_karpathy_test.json','r'))
ls = ['/root/fiftyone/coco-2014/validation/data/'+ann[i]['image'].split('/')[-1] for i in range(len(ann))]

category = ['dining table', 'bed']
all_images_view = dataset.match(F("filepath").is_in(set(ls))).select_fields("ground_truth")
big_images_view = (dataset.match(F("filepath").is_in(set(ls)))
                            .select_fields("ground_truth")
                            .filter_labels("ground_truth", (bbox_area >= 0.85) & (bbox_area < 1))
                            .filter_labels("ground_truth", (bbox_area0 <= 0.01) | (bbox_area1 <= 0.01) | (bbox_area0 + bbox_area2 >= 0.99) | (bbox_area1 + bbox_area3 >= 0.99))
                            .sort_by(bbox_area, reverse=True)
                            .group_by("filepath")
                            )

print('len big_images_view: {}'.format(len(big_images_view)))

session.view = big_images_view

## select 200 images for subset
big_images_view = big_images_view.limit(200)
from collections import Counter
counter_all = Counter()
counter = Counter()
for sample in big_images_view:
    labels = set([label["label"] for label in sample.ground_truth.detections])
    counter.update(labels)

sorted_dict = dict(sorted(counter.items(), key=lambda x: x[1], reverse=True))
print(sorted_dict)

for sample in all_images_view:
  if sample.ground_truth is not None:
    labels = set([label["label"] for label in sample.ground_truth.detections])
    counter_all.update(labels)

sorted_dict_all = dict(sorted(counter_all.items(), key=lambda x: x[1], reverse=True))
print(sorted_dict_all)

ratio = {k: sorted_dict[k]/sorted_dict_all[k] for k in sorted_dict}
print(ratio)

# save json file
images = []
for sample in big_images_view:
      images.append(sample.filepath.split('/')[-1])
print(len(images))

with open(os.path.join("coco_karpathy_scale_0_85_1_colabocc_images.json"),"w") as f:
    f.write(json.dumps(images))


--2026-02-04 17:19:33--  https://storage.googleapis.com/sfr-vision-language-research/datasets/coco_karpathy_test.json
Resolving storage.googleapis.com (storage.googleapis.com)... 74.125.135.207, 142.251.188.207, 192.178.163.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|74.125.135.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1735468 (1.7M) [application/json]
Saving to: ‘coco_karpathy_test.json’

coco_karpathy_test. 100%[===================>]   1.65M  --.-KB/s    in 0.02s   

2026-02-04 17:19:33 (95.2 MB/s) - ‘coco_karpathy_test.json’ saved [1735468/1735468]

len big_images_view: 341
{'dining table': 73, 'person': 35, 'bed': 12, 'bowl': 6, 'oven': 6, 'car': 6, 'refrigerator': 6, 'pizza': 5, 'chair': 5, 'couch': 5, 'train': 5, 'bench': 4, 'suitcase': 3, 'potted plant': 3, 'truck': 3, 'laptop': 3, 'bird': 2, 'boat': 2, 'cat': 2, 'elephant': 2, 'keyboard': 2, 'dog': 2, 'cake': 2, 'orange': 1, 'airplane': 1, 'book': 1, 'giraffe': 1, 'tennis racket': 1, 'parking meter': 1, 'umbrella': 1, 'donut': 1, 'bus': 1, 'toilet': 1}
{'person': 2627, 'chair': 545, 'car': 522, 'dining table': 517, 'cup': 382, 'bottle': 369, 'handbag': 288, 'bowl': 286, 'truck': 273, 'book': 247, 'bench': 232, 'clock': 213, 'backpack': 212, 'sink': 205, 'cell phone': 199, 'traffic light': 192, 'cat': 192, 'dog': 183, 'knife': 182, 'couch': 175, 'sports ball': 170, 'potted plant': 167, 'tv': 167, 'umbrella': 165, 'tie': 161, 'spoon': 161, 'toilet': 159, 'surfboard': 156, 'laptop': 150, 'bus': 148, 'bed': 148, 'motorcycle': 148, 'train': 146, 'fork': 143, 'skateboard': 143, 'boat': 139, 'bird': 131, 'pizza': 131, 'cake': 128, 'wine glass': 128, 'tennis racket': 127, 'bicycle': 126, 'remote': 125, 'vase': 120, 'skis': 118, 'horse': 118, 'giraffe': 113, 'airplane': 113, 'baseball glove': 110, 'suitcase': 110, 'oven': 108, 'sandwich': 106, 'baseball bat': 98, 'refrigerator': 94, 'elephant': 93, 'teddy bear': 89, 'kite': 87, 'frisbee': 85, 'keyboard': 85, 'cow': 84, 'mouse': 82, 'banana': 80, 'zebra': 78, 'donut': 77, 'fire hydrant': 77, 'broccoli': 75, 'stop sign': 75, 'carrot': 75, 'orange': 74, 'sheep': 73, 'apple': 69, 'snowboard': 65, 'microwave': 54, 'hot dog': 51, 'scissors': 43, 'toothbrush': 43, 'parking meter': 37, 'bear': 29, 'toaster': 11, 'hair drier': 9}
{'dining table': 0.14119922630560927, 'person': 0.013323182337266844, 'bed': 0.08108108108108109, 'bowl': 0.02097902097902098, 'oven': 0.05555555555555555, 'car': 0.011494252873563218, 'refrigerator': 0.06382978723404255, 'pizza': 0.03816793893129771, 'chair': 0.009174311926605505, 'couch': 0.02857142857142857, 'train': 0.03424657534246575, 'bench': 0.017241379310344827, 'suitcase': 0.02727272727272727, 'potted plant': 0.017964071856287425, 'truck': 0.01098901098901099, 'laptop': 0.02, 'bird': 0.015267175572519083, 'boat': 0.014388489208633094, 'cat': 0.010416666666666666, 'elephant': 0.021505376344086023, 'keyboard': 0.023529411764705882, 'dog': 0.01092896174863388, 'cake': 0.015625, 'orange': 0.013513513513513514, 'airplane': 0.008849557522123894, 'book': 0.004048582995951417, 'giraffe': 0.008849557522123894, 'tennis racket': 0.007874015748031496, 'parking meter': 0.02702702702702703, 'umbrella': 0.006060606060606061, 'donut': 0.012987012987012988, 'bus': 0.006756756756756757, 'toilet': 0.006289308176100629}
200

In [ ]:
# import fiftyone as fo
# import fiftyone.zoo as foz
# from fiftyone import ViewField as F
# import json
# import numpy as np
# import os

# bbox_area = F("bounding_box")[2] * F("bounding_box")[3]
# bbox_area0 = F("bounding_box")[0]
# bbox_area1 = F("bounding_box")[1]
# bbox_area2 = F("bounding_box")[2]
# bbox_area3 = F("bounding_box")[3]
# !wget https://storage.googleapis.com/sfr-vision-language-research/datasets/coco_karpathy_test.json
# ann = json.load(open('coco_karpathy_test.json','r'))
# ls = ['/root/fiftyone/coco-2014/validation/data/'+ann[i]['image'].split('/')[-1] for i in range(len(ann))]

# category = ['person','cat']

# big_images_view = (dataset.match(F("filepath").is_in(set(ls)))
#                             .select_fields("ground_truth")
#                             .filter_labels("ground_truth", (bbox_area >= 0.15) & (bbox_area < 1))
#                             .sort_by(bbox_area, reverse=True)
#                             )

# small_images_view = (dataset.match(F("filepath").is_in(set(ls)))
#                             .select_fields("ground_truth")
#                             .filter_labels("ground_truth", (bbox_area >= 0.0) & (bbox_area < 0.15))
#                             .filter_labels("ground_truth", (bbox_area0 <= 0.01) | (bbox_area1 <= 0.01) | (bbox_area2 >= 0.99) | (bbox_area3 >= 0.99))
#                             .sort_by(bbox_area, reverse=True)
#                             .exclude(big_images_view)
#                             # .filter_labels("ground_truth", F("label").is_in(set(category)))
#                             )
# print(len(small_images_view))
# session.view = small_images_view

# # save json file
# images = []
# for sample in small_images_view:
#       images.append(sample.filepath.split('/')[-1])
# print(len(images))

# with open(os.path.join("coco_karpathy_scale_0_0.15_colabocc.json"),"a") as f:
#     f.write(json.dumps(images))
